In [ ]:
import numpy as np
import pandas as pd
!pip install datatable > /dev/null
import optuna
import datatable as dt
import os
import pickle

In [ ]:
!nvidia-smi

### Pickle training data

In [ ]:
# import datatable as dt

# df_dt = dt.fread("../input/jane-street-market-prediction/train.csv")
# df = df_dt.to_pandas()

# # Quick cleaning
# SEED = 1111
# np.random.seed(SEED)

# df = df.query('date > 85').reset_index(drop = True) 
# df = df[df['weight'] != 0]
# df.fillna(df.mean(),inplace=True)
# df['action'] = ((df['resp'].values) > 0).astype(int)

# df.reset_index(drop=True, inplace=True)

# df = df.astype({c: np.float32 for c in df.select_dtypes(include='float64').columns})

# pickle.dump(df, open('/kaggle/working/train.csv.pandas.pickle', 'wb'))
# df.shape

In [ ]:
import datatable as dt

print("Loading data")

df_dt = dt.fread("../input/jane-street-market-prediction/train.csv")
df = df_dt.to_pandas()

features = [c for c in df.columns if 'feature' in c]

print("Filling missing values")

f_mean = df[features[1:]].mean()

df  = df.loc[df.weight > 0].reset_index(drop=True)
df[features[1:]] = df[features[1:]]. fillna(f_mean)
df['action'] = (df['resp'] > 0).astype('int')

df = df.astype({c: np.float32 for c in df.select_dtypes(include='float64').columns})

f_mean = f_mean.values
np.save('/kaggle/working/f_mean.npy', f_mean)

pickle.dump(df, open('/kaggle/working/train.csv.pandas.pickle', 'wb'))
df.shape

In [ ]:
del df
import gc
gc.collect()

### Read in pickled file

In [ ]:
%%time
train_pickle_file = '/kaggle/working/train.csv.pandas.pickle'
df = pickle.load(open(train_pickle_file, 'rb'))
df.shape

### Create folds using GroupTimeSeriesSplit

In [ ]:
# # create_folds.py
# import pandas as pd
# from sklearn.model_selection import KFold

# # Pass in just target dataframe for the training set
# def create_folds(df):
#     df.loc[:,"kfold"] = -1
    
#     kf = KFold(n_splits=5, random_state=101, shuffle=True)

#     for fold, (trn, val) in enumerate(kf):
#         df.loc[val,"kfold"] = fold
    
#     df.to_csv("../input/jane-street-market-prediction/train_folds.csv", index=False)

In [ ]:
# GroupTimeSeriesSplit.py
from sklearn.model_selection._split import _BaseKFold, indexable, _num_samples
from sklearn.utils.validation import _deprecate_positional_args

# https://github.com/getgaurav2/scikit-learn/blob/d4a3af5cc9da3a76f0266932644b884c99724c57/sklearn/model_selection/_split.py#L2243
class GroupTimeSeriesSplit(_BaseKFold):
    """Time Series cross-validator variant with non-overlapping groups.
    Provides train/test indices to split time series data samples
    that are observed at fixed time intervals according to a
    third-party provided group.
    In each split, test indices must be higher than before, and thus shuffling
    in cross validator is inappropriate.
    This cross-validation object is a variation of :class:`KFold`.
    In the kth split, it returns first k folds as train set and the
    (k+1)th fold as test set.
    The same group will not appear in two different folds (the number of
    distinct groups has to be at least equal to the number of folds).
    Note that unlike standard cross-validation methods, successive
    training sets are supersets of those that come before them.
    Read more in the :ref:`User Guide <cross_validation>`.
    Parameters
    ----------
    n_splits : int, default=5
        Number of splits. Must be at least 2.
    max_train_size : int, default=None
        Maximum size for a single training set.
    Examples
    --------
    >>> import numpy as np
    >>> from sklearn.model_selection import GroupTimeSeriesSplit
    >>> groups = np.array(['a', 'a', 'a', 'a', 'a', 'a',\
                           'b', 'b', 'b', 'b', 'b',\
                           'c', 'c', 'c', 'c',\
                           'd', 'd', 'd'])
    >>> gtss = GroupTimeSeriesSplit(n_splits=3)
    >>> for train_idx, test_idx in gtss.split(groups, groups=groups):
    ...     print("TRAIN:", train_idx, "TEST:", test_idx)
    ...     print("TRAIN GROUP:", groups[train_idx],\
                  "TEST GROUP:", groups[test_idx])
    TRAIN: [0, 1, 2, 3, 4, 5] TEST: [6, 7, 8, 9, 10]
    TRAIN GROUP: ['a' 'a' 'a' 'a' 'a' 'a']\
    TEST GROUP: ['b' 'b' 'b' 'b' 'b']
    TRAIN: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10] TEST: [11, 12, 13, 14]
    TRAIN GROUP: ['a' 'a' 'a' 'a' 'a' 'a' 'b' 'b' 'b' 'b' 'b']\
    TEST GROUP: ['c' 'c' 'c' 'c']
    TRAIN: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]\
    TEST: [15, 16, 17]
    TRAIN GROUP: ['a' 'a' 'a' 'a' 'a' 'a' 'b' 'b' 'b' 'b' 'b' 'c' 'c' 'c' 'c']\
    TEST GROUP: ['d' 'd' 'd']
    """
    @_deprecate_positional_args
    def __init__(self,
                 n_splits=5,
                 *,
                 max_train_size=None
                 ):
        super().__init__(n_splits, shuffle=False, random_state=None)
        self.max_train_size = max_train_size

    def split(self, X, y=None, groups=None):
        """Generate indices to split data into training and test set.
        Parameters
        ----------
        X : array-like of shape (n_samples, n_features)
            Training data, where n_samples is the number of samples
            and n_features is the number of features.
        y : array-like of shape (n_samples,)
            Always ignored, exists for compatibility.
        groups : array-like of shape (n_samples,)
            Group labels for the samples used while splitting the dataset into
            train/test set.
        Yields
        ------
        train : ndarray
            The training set indices for that split.
        test : ndarray
            The testing set indices for that split.
        """
        if groups is None:
            raise ValueError(
                "The 'groups' parameter should not be None")
        X, y, groups = indexable(X, y, groups)
        n_samples = _num_samples(X)
        n_splits = self.n_splits
        n_folds = n_splits + 1
        group_dict = {}
        u, ind = np.unique(groups, return_index=True)
        unique_groups = u[np.argsort(ind)]
        n_samples = _num_samples(X)
        n_groups = _num_samples(unique_groups)
        for idx in np.arange(n_samples):
            if (groups[idx] in group_dict):
                group_dict[groups[idx]].append(idx)
            else:
                group_dict[groups[idx]] = [idx]
        if n_folds > n_groups:
            raise ValueError(
                ("Cannot have number of folds={0} greater than"
                 " the number of groups={1}").format(n_folds,
                                                     n_groups))
        group_test_size = n_groups // n_folds
        group_test_starts = range(n_groups - n_splits * group_test_size,
                                  n_groups, group_test_size)
        for group_test_start in group_test_starts:
            train_array = []
            test_array = []
            for train_group_idx in unique_groups[:group_test_start]:
                train_array_tmp = group_dict[train_group_idx]
                train_array = np.sort(np.unique(
                                      np.concatenate((train_array,
                                                      train_array_tmp)),
                                      axis=None), axis=None)
            train_end = train_array.size
            if self.max_train_size and self.max_train_size < train_end:
                train_array = train_array[train_end -
                                          self.max_train_size:train_end]
            for test_group_idx in unique_groups[group_test_start:
                                                group_test_start +
                                                group_test_size]:
                test_array_tmp = group_dict[test_group_idx]
                test_array = np.sort(np.unique(
                                              np.concatenate((test_array,
                                                              test_array_tmp)),
                                     axis=None), axis=None)
            yield [int(i) for i in train_array], [int(i) for i in test_array]

            
def create_folds(df):
#     df.loc[:,"kfold"] = -1
    
    for fold,(train_idx, valid_idx) in enumerate(GroupTimeSeriesSplit().split(df, groups=df['date'])):
        train_temp = df.loc[train_idx,:]
        valid_temp = df.loc[valid_idx,:]
        train_temp.to_csv(f"/kaggle/working/train_fold{fold+1}.csv", index=False)
        valid_temp.to_csv(f"/kaggle/working/valid_fold{fold+1}.csv", index=False)
        
    print("Folds created")


temp = df[['ts_id','date']].copy()
create_folds(temp)

In [ ]:
del df
import gc
gc.collect()

### Model training

In [ ]:
!ls -hlt /kaggle/working/

In [ ]:
# utils.py
import torch
import torch.nn as nn

class JaneStreetDataset:
    def __init__(self, features, targets):
        self.features = features
        self.targets = targets.reshape(-1, 1)
    
    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, item):
        return {
            "x": torch.tensor(self.features[item, :], dtype=torch.float),
            "y": torch.tensor(self.targets[item, :], dtype=torch.float)
        }

    
class Engine:
    def __init__(self, model, optimizer, device):
        self.model = model
        self.device = device
        self.optimizer = optimizer
    
    @staticmethod
    def loss_fn(targets, outputs):
        return nn.BCEWithLogitsLoss()(outputs, targets)
    
    def train(self, data_loader):
        self.model.train()
        final_loss=0
        for data in data_loader:
            self.optimizer.zero_grad()
            inputs = data["x"].to(self.device)
            targets = data["y"].to(self.device)
            outputs = self.model(inputs)
            loss = self.loss_fn(targets, outputs)
            loss.backward()
            self.optimizer.step()
            final_loss += loss.item()
        return final_loss/ len(data_loader)
    
    def evaluate(self, data_loader):
        self.model.eval()
        final_loss=0
        for data in data_loader:
            inputs = data["x"].to(self.device)
            targets = data["y"].to(self.device)
            outputs = self.model(inputs)
            loss = self.loss_fn(targets, outputs)
            final_loss += loss.item()
        return final_loss/ len(data_loader)

    
class Model(nn.Module):
    def __init__(self, nfeatures, ntargets, nlayers, hidden_size, dropout):
        super().__init__()
        layers = []
        for _ in range(nlayers):
            if len(layers) == 0:
                layers.append(nn.Linear(nfeatures, hidden_size))
                layers.append(nn.BatchNorm1d(hidden_size))
                layers.append(nn.Dropout(dropout))
                layers.append(nn.ReLU())
            else:
                layers.append(nn.Linear(hidden_size, hidden_size))
                layers.append(nn.BatchNorm1d(hidden_size))
                layers.append(nn.Dropout(dropout))
                layers.append(nn.ReLU())
        layers.append(nn.Linear(hidden_size, ntargets))
        self.model = nn.Sequential(*layers)
    
    
    def forward(self, x):
        return self.model(x)
                

In [ ]:
# train.py
import torch
import pandas as pd
import numpy as np
import datatable as dt
import pickle
import gc

DEVICE = torch.device('cuda:0')
EPOCHS = 100

def run_training(fold, params, save_model=False):
    train_pickle_file = '/kaggle/working/train.csv.pandas.pickle'
    df = pickle.load(open(train_pickle_file, 'rb'))
    
    print("Dataset loaded")
    
    # Read in training folds
    train_fold = pd.read_csv(f"/kaggle/working/train_fold{fold+1}.csv")
    valid_fold = pd.read_csv(f"/kaggle/working/valid_fold{fold+1}.csv")

    feature_cols = [c for c in df.columns if "feature" in c]
    
    # Train and validation folds
    train_df = pd.merge(df, train_fold, on="ts_id", how="inner")
    valid_df = pd.merge(df, valid_fold, on="ts_id", how="inner")
    
    del df
    gc.collect()
    
    print("Folds loaded")
    
    xtrain = train_df[feature_cols].to_numpy()
    ytrain = train_df["action"].to_numpy()
    
    xvalid = valid_df[feature_cols].to_numpy()
    yvalid = valid_df["action"].to_numpy()
    
    train_dataset = JaneStreetDataset(features=xtrain, targets=ytrain)
    valid_dataset = JaneStreetDataset(features=xvalid, targets=yvalid)
    
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1024, num_workers=4)
    valid_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1024, num_workers=4)
    
    del train_df, valid_df
    gc.collect()
    
    model = Model(
        nfeatures=np.shape(xtrain)[1], 
        ntargets=1, 
        nlayers=params['num_layers'], 
        hidden_size=params['hidden_size'],
        dropout=params['dropout']
    )
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
    eng = Engine(model, optimizer, device=DEVICE)
    
    best_loss = np.inf
    early_stopping_iter = 10
    early_stopping_counter = 0
    
    print("Model training begins")
    
    for epoch in range(EPOCHS):
        train_loss = eng.train(train_loader)
        valid_loss = eng.evaluate(valid_loader)
        print(f"{fold}, {epoch}, {train_loss}, {valid_loss}")
        if valid_loss < best_loss:
            best_loss = valid_loss
            if save_model:
                torch.save(model.state_dict(), f"/kaggle/working/model_fold_{fold+1}.bin")
        else:
            early_stopping_counter+=1
        
        if early_stopping_counter > early_stopping_iter:
            break
    
    return best_loss



def objective(trial):
    params = {
        "num_layers": trial.suggest_int("num_layer", 1, 3),
        "hidden_size": trial.suggest_int("hidden_size", 16, 512),
        "dropout": trial.suggest_uniform("dropout", 0.1, 0.7),
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-6, 1e-3)
    }
    all_losses = []
    for f_ in range(5):
        temp_loss = run_training(f_, params, save_model=False)
        all_losses.append(temp_loss)
    
    return np.mean(all_losses)

In [ ]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print("Best trial:")
trial_ = study.best_trial

print(trial_.values)
print(trial_.params)

scores = 0
for j in range(5):
    scr = run_training(j, trial_.params, save_model=True)
    scores+=scr

print(scores/5)